In [1]:
import tensorflow as tf
from tqdm import tqdm_notebook

print(tf.config.list_physical_devices('GPU'))

C:\Users\PC\.conda\envs\recora\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ============================================================
# 0) INSTALL
# ============================================================
# pip install -U LibRecommender tensorflow pandas numpy clickhouse-connect scikit-learn

# If you are in notebook / interactive env and hit TF graph reuse issues:
# import tensorflow as tf
# tf.compat.v1.reset_default_graph()


# ============================================================
# 1) IMPORTS
# ============================================================
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import clickhouse_connect
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer

from recora.data import DatasetFeat

import pandas as pd

from recora.data import DatasetFeat, split_by_ratio_chrono, random_split

TRAIN_DAYS = 90
RANDOM_SEED = 42
TOP_K = 5

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

warnings.filterwarnings("ignore")

In [19]:
# # ============================================================
# # 2) GLOBAL CONFIG
# # ============================================================
# CLICKHOUSE_CONFIG = {
# 	"host": os.getenv("CH_HOST", "172.30.36.13"),
# 	"port": int(os.getenv("CH_PORT", "8123")),
# 	"username": os.getenv("CH_USER", "default"),
# 	"password": os.getenv("CH_PASSWORD", "192837465AhurA!@#"),
# 	"database": os.getenv("CH_DATABASE", "KF"),
# 	"secure": os.getenv("CH_SECURE", "false").lower() == "true",
# }
#
# # ============================================================
# # 3) CLICKHOUSE CLIENT
# # ============================================================
# client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)
#
# # ============================================================
# # 4) SQL QUERIES
# #    KEEPING YOUR SQL UNCHANGED
# # ============================================================
# ITEM_SQL = f"""
# WITH item_base AS (SELECT pp.category_level2_id AS category_id,
#                           pp.category_level1_id AS root_category_id
#                    FROM KF.plane_products pp
#                    WHERE pp.category_level1_activity_type_id IN (1, 29)
#                      AND pp.category_level2_activity_type_id IN (1, 29)
#                      AND pp.product_id <> 0
#                    GROUP BY pp.category_level2_id, pp.category_level1_id),
#
#      availability_price AS (SELECT pp.category_level2_id          AS category_id,
#                                    avg(isph.availability)         AS avg_availability,
#                                    stddevSamp(isph.availability)  AS std_availability,
#
#                                    avg(isph.avg_price)            AS avg_price,
#                                    stddevSamp(isph.avg_price)     AS std_price,
#                                    min(isph.avg_price)            AS min_price,
#                                    max(isph.avg_price)            AS max_price,
#                                    quantile(0.25)(isph.avg_price) AS p25_price,
#                                    quantile(0.5)(isph.avg_price)  AS p50_price, -- corrected: median
#                                    quantile(0.75)(isph.avg_price) AS p75_price
#                             FROM KF.in_stock_price_history isph
#                                      INNER JOIN KF.plane_products pp ON isph.shop_product_id = pp.shop_product_id
#                             WHERE isph.snapshot_date > today() - {TRAIN_DAYS}
#                               AND pp.category_level1_activity_type_id IN (1, 29)
#                               AND pp.category_level2_activity_type_id IN (1, 29)
#                               AND pp.product_id <> 0
#                             GROUP BY pp.category_level2_id),
#
#      popularity_discount AS (SELECT pp.category_level2_id                                        AS category_id,
#
#                                     count(*)                                                     AS total_popularity,
#                                     countIf(oi.order_date > today() - 30)                        AS recent_popularity,
#                                     recent_popularity / total_popularity                         AS popularity_trend,
#
#                                     sum(oi.items_jet_discount + oi.brand_discount + oi.items_vendor_discount) /
#                                     nullIf(sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount +
#                                                oi.brand_discount + oi.items_vendor_discount), 0) AS discount_ratio,
#
#                                     topK(1)(oi.order_hour)[1]                                    AS mode_ordering_hour
#                              FROM KF.fact_order_items oi
#                                       INNER JOIN KF.plane_products pp ON oi.shop_product_id = pp.shop_product_id
#                              WHERE oi.is_gross = 1
#                                AND oi.order_date > today() - {TRAIN_DAYS}
#                                AND pp.category_level1_activity_type_id IN (1, 29)
#                                AND pp.category_level2_activity_type_id IN (1, 29)
#                                AND pp.product_id <> 0
#                              GROUP BY pp.category_level2_id),
#
#      product_seller_stats AS (SELECT pp.category_level2_id              AS category_id,
#                                      countDistinct(pp.shop_product_id)  AS num_products,
#                                      countDistinct(pp.merchant_shop_id) AS num_sellers
#                               FROM KF.plane_products pp
#                               WHERE pp.category_level1_activity_type_id IN (1, 29)
#                                 AND pp.category_level2_activity_type_id IN (1, 29)
#                                 AND pp.product_id <> 0
#                               GROUP BY pp.category_level2_id),
#
#      first_appearance AS (SELECT pp.category_level2_id   AS category_id,
#                                  min(isph.snapshot_date) AS first_seen_date
#                           FROM KF.in_stock_price_history isph
#                                    INNER JOIN KF.plane_products pp ON isph.shop_product_id = pp.shop_product_id
#                           WHERE pp.category_level1_activity_type_id IN (1, 29)
#                             AND pp.category_level2_activity_type_id IN (1, 29)
#                             AND pp.product_id <> 0
#                           GROUP BY pp.category_level2_id)
#
# /*----- main SELECT: include all features -----*/
# SELECT toString(b.category_id)                      AS category_id,
#        toString(b.root_category_id)                 AS root_category_id,
#
#        -- availability & price distribution
#        ap.avg_availability,
#        ap.std_availability,
#        ap.avg_price,
#        ap.std_price,
#        ap.min_price,
#        ap.max_price,
#        ap.p25_price,
#        ap.p50_price,
#        ap.p75_price,
#
#        -- popularity & discount
#        p.total_popularity,
#        p.recent_popularity,
#        p.popularity_trend,
#        p.discount_ratio                             AS discount_ratio,
#        p.mode_ordering_hour                         AS mode_ordering_hour,
#
#        -- product/seller variety
#        ps.num_products,
#        ps.num_sellers,
#
#        -- item age (days since first appearance)
#        dateDiff('day', fa.first_seen_date, today()) AS item_age_days
# FROM item_base b
#          JOIN availability_price ap
#               ON b.category_id = ap.category_id
#          JOIN popularity_discount p
#               ON b.category_id = p.category_id
#          JOIN product_seller_stats ps
#               ON b.category_id = ps.category_id
#          JOIN first_appearance fa
#               ON b.category_id = fa.category_id
# """
#
# USER_SQL = f"""
# /*----- CTEs for item features -----*/
# WITH
# -- per‑user basic stats ------------------------------------------------
# user_base AS (SELECT oi.user_id                                                                           AS user_id,
#                      nullIf(argMax(ua.polygon_id, ua.address_id), 0)                                      AS polygon_id,
#
#                      dateDiff('day', max(oi.order_date), today())                                         AS recency,
#                      dateDiff('day', min(oi.order_date), today())                                         AS length,
#                      countDistinct(oi.order_id)                                                           AS frequency,
#                      sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS monetary,
#
#                      countDistinct(pp.brand_id)                                                           AS brand_diversity,
#                      countDistinct(pp.category_level2_id)                                                 AS category_diversity,
#                      countDistinct(pp.category_level1_id)                                                 AS root_category_diversity,
#
#                      monetary / frequency                                                                 AS aov,
#                      count(oi.item_id) / frequency                                                        AS lpo, -- lines per order (number of item rows)
#                      sum(oi.quantity) / countDistinct(oi.order_id)                                        AS ipo, -- items per order (total quantity)
#
#                      avg(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS avg_item_spending,
#                      min(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS min_item_spending,
#                      max(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS max_item_spending,
#                      quantile(0.25)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount) AS p25_item_spending,
#                      quantile(0.5)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)  AS p50_item_spending,
#                      quantile(0.75)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount) AS p75_item_spending,
#
#                      sum(oi.items_jet_discount + oi.brand_discount + oi.items_vendor_discount) /
#                      nullIf(sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount +
#                                 oi.brand_discount + oi.items_vendor_discount), 0)                         AS discount_ratio,
#
#                      avg(oi.order_hour)                                                                   AS avg_ordering_hour
#
#               FROM KF.fact_order_items AS oi
#                        JOIN KF.dim_merchant_shops AS ms ON oi.merchant_shop_id = ms.merchant_shop_id
#                        JOIN KF.dim_user_addresses AS ua ON oi.address_id = ua.address_id
#                        JOIN KF.plane_products AS pp ON oi.shop_product_id = pp.shop_product_id
#               WHERE oi.is_gross = 1
#                 AND oi.order_date > today() - {TRAIN_DAYS}
#                 AND ms.activity_type_id IN (1, 29)
#                 AND pp.category_level1_activity_type_id IN (1, 29)
#                 AND pp.category_level2_activity_type_id IN (1, 29)
#               GROUP BY oi.user_id)
#
# SELECT ub.user_id           AS user_id,
#        ub.polygon_id,
#        ub.recency,
#        ub.frequency,
#        ub.monetary,
#        ub.length,
#
#        ub.brand_diversity,
#        ub.category_diversity,
#        ub.root_category_diversity,
#
#        ub.avg_ordering_hour AS user_avg_ordering_hour,
#
#        ub.ipo,
#        ub.lpo,
#        ub.aov,
#
#        ub.avg_item_spending,
#        ub.min_item_spending,
#        ub.max_item_spending,
#        ub.p25_item_spending,
#        ub.p50_item_spending,
#        ub.p75_item_spending,
#
#        ub.discount_ratio    AS user_discount_ratio
# FROM user_base AS ub
# """
#
# INTERACTION_SQL = f"""
# SELECT oi.user_id                                                    AS user_id,
#        pp.category_level2_id                                         AS category_id,
#        -- H: 14 days
#        exp(-log(2) / 14.0 * dateDiff('day', oi.order_date, today())) AS sample_weight,
#        -dateDiff('day', oi.order_date, today())                      AS time,
#        oi.order_hour                                                 AS order_hour
# FROM KF.fact_order_items oi
#          INNER JOIN KF.plane_products pp
#                     ON oi.shop_product_id = pp.shop_product_id
# WHERE oi.is_gross = 1
#   AND oi.order_date > today() - {TRAIN_DAYS}
#   AND pp.category_level1_activity_type_id IN (1, 29)
#   AND pp.category_level2_activity_type_id IN (1, 29)
#   AND pp.product_id != 0
# GROUP BY oi.order_date, oi.order_hour, oi.user_id, pp.category_level2_id
# """
#
# # # ============================================================
# # # 5) READ DATA
# # # ============================================================
# item_df = client.query_df(ITEM_SQL)
# print("item_df shape      :", item_df.shape)
#
# user_df = client.query_df(USER_SQL)
# print("user_df shape      :", user_df.shape)
#
# inter_events = client.query_df(INTERACTION_SQL)
# print("inter_events shape :", inter_events.shape)

In [20]:
# item_df.to_csv('./sample_data/recora_category_item_df.csv')
# user_df.to_csv('./sample_data/recora_category_user_df.csv')
# inter_events.to_csv('./sample_data/recora_category_inter_events.csv')

In [21]:
item_df = pd.read_csv('./sample_data/recora_category_item_df.csv', index_col=0).rename(columns={'category_id': 'item'})
user_df = pd.read_csv('./sample_data/recora_category_user_df.csv', index_col=0).rename(columns={'user_id': 'user'})
inter_events = pd.read_csv('./sample_data/recora_category_inter_events.csv', index_col=0).rename(
	columns={'category_id': 'item', 'user_id': 'user'})

In [22]:
print("item_df features:\n", item_df.columns.tolist(), "\n")
print("user_df features:\n", user_df.columns.tolist(), "\n")
print("inter_events features:\n", inter_events.columns.tolist(), "\n")

item_df features:
 ['item', 'root_category_id', 'avg_availability', 'std_availability', 'avg_price', 'std_price', 'min_price', 'max_price', 'p25_price', 'p50_price', 'p75_price', 'total_popularity', 'recent_popularity', 'popularity_trend', 'discount_ratio', 'mode_ordering_hour', 'num_products', 'num_sellers', 'item_age_days'] 

user_df features:
 ['user', 'polygon_id', 'recency', 'frequency', 'monetary', 'length', 'brand_diversity', 'category_diversity', 'root_category_diversity', 'user_avg_ordering_hour', 'ipo', 'lpo', 'aov', 'avg_item_spending', 'min_item_spending', 'max_item_spending', 'p25_item_spending', 'p50_item_spending', 'p75_item_spending', 'user_discount_ratio'] 

inter_events features:
 ['user', 'item', 'sample_weight', 'time', 'order_hour'] 



In [23]:
pareto = inter_events.groupby('item').size().rename("pareto").sort_values(ascending=False).cumsum()
pareto = pareto / pareto.max()
pareto = pareto[pareto <= 0.95].reset_index()
pareto

,item,pareto
0,69,0.051817
1,33,0.103089
2,3070,0.144626
3,3097,0.184183
4,3100,0.222601
...,...,...
64,54,0.935802
65,22,0.939072
66,3139,0.942143
67,3129,0.945052


In [24]:
filtered_users = inter_events.groupby('user')['item'].count().reset_index()
filtered_users = filtered_users[filtered_users['item'] > 0]

inter_events = inter_events.loc[
	(inter_events['user'].isin(filtered_users['user'])) &
	(inter_events['item'].isin(pareto['item']))
	]

inter_events['user'] = inter_events['user'].astype(str)
inter_events['item'] = inter_events['item'].astype(str)

item_df['item'] = item_df['item'].astype(str)
user_df['user'] = user_df['user'].astype(str)

In [25]:
inter_events = inter_events.sort_values(by='time', ascending=True)

In [324]:
from recora.data import split_by_num, random_split, split_by_ratio, split_by_ratio_chrono

# train_data, eval_data = split_by_ratio_chrono(inter_events, order=True, shuffle=False, test_size=0.1)
# train_data, eval_data = split_by_num(df, shuffle=True, order=False, test_size=5)
train_inter_events, eval_inter_events = random_split(inter_events, shuffle=False, test_size=0.1)

In [325]:
# Single sparse columns
single_sparse_feature_columns = [
	"root_category_id",
	"polygon_id"
]

# Multi sparse columns (list of lists for multi‑hot encoding)
multi_sparse_feature_columns = [
	[]
]

# All dense features (both user and item sides, plus order_hour)
dense_feature_columns = [
	# Item dense features
	'avg_availability', 'std_availability',
	'avg_price', 'std_price', 'min_price', 'max_price', 'p25_price',
	'p50_price', 'p75_price', 'total_popularity', 'recent_popularity',
	'popularity_trend', 'discount_ratio', 'mode_ordering_hour',
	'num_products', 'num_sellers', 'item_age_days',

	# User dense features
	"recency", "length", "frequency", "monetary",
	"brand_diversity", "category_diversity", "root_category_diversity",
	"user_avg_ordering_hour", "ipo", "lpo", "aov",
	"avg_item_spending", "min_item_spending", "max_item_spending",
	"p25_item_spending", "p50_item_spending", "p75_item_spending",
	"user_discount_ratio",

	# Interevents dense
	"time", "order_hour"
]

# Features belonging to the user side
user_feature_columns = [
	# User sparse
	"polygon_id",

	# User dense
	'recency', 'frequency', 'monetary', 'length',
	'brand_diversity', 'category_diversity', 'root_category_diversity',
	'user_avg_ordering_hour', 'ipo', 'lpo', 'aov', 'avg_item_spending',
	'min_item_spending', 'max_item_spending', 'p25_item_spending',
	'p50_item_spending', 'p75_item_spending', 'user_discount_ratio',

	# Interevents dense
	"time", "order_hour"
]

# Features belonging to the item side
item_feature_columns = [
	# Item sparse
	"root_category_id",

	# Item dense
	'avg_availability', 'std_availability',
	'avg_price', 'std_price', 'min_price', 'max_price', 'p25_price',
	'p50_price', 'p75_price', 'total_popularity', 'recent_popularity',
	'popularity_trend', 'discount_ratio', 'mode_ordering_hour',
	'num_products', 'num_sellers', 'item_age_days',
]

In [326]:
train_inter_events = train_inter_events.groupby(['user', 'item']).agg({
	# 'sample_weight': 'sum',  # total strength
    'sample_weight': 'count',  # total strength

	'time': 'max',  # most recent interaction
	'order_hour': 'mean'  # most frequent hour
}).reset_index()

In [327]:
eval_inter_events = eval_inter_events.groupby(['user', 'item']).agg({
	# 'sample_weight': 'sum',  # total strength
    'sample_weight': 'count',  # total strength

	'time': 'max',  # most recent interaction
	'order_hour': 'mean'  # most frequent hour        # typical hour
}).reset_index()

In [328]:
# train data
train_data = (
	train_inter_events.
	merge(user_df, how="inner", on="user", ).
	merge(item_df, how="inner", on="item", )
)

train_data['label'] = 1

train_data = train_data[['user', 'item', 'label', 'sample_weight'] + user_feature_columns + item_feature_columns]
train_data = train_data.fillna(0)

# eval data
eval_data = (
	eval_inter_events.
	merge(user_df, how="inner", on="user", ).
	merge(item_df, how="inner", on="item", )
)

eval_data['label'] = 1

eval_data = eval_data[['user', 'item', 'label'] + user_feature_columns + item_feature_columns]
eval_data = eval_data.fillna(0)

In [329]:
train_data['time'].min(), train_data['time'].max(), eval_data['time'].min(), eval_data['time'].max()

(-89, -14, -14, -1)

In [330]:
# Handle single sparse columns
for c in single_sparse_feature_columns:
	for dataframe in [train_data, eval_data]:
		dataframe[c] = dataframe[c].astype(str).fillna("missing")
		dataframe.loc[dataframe[c] == "0", c] = "missing"

# # Handle multi sparse columns
# multi_sparse_flat = [col for group in multi_sparse_col for col in group]
# for c in multi_sparse_flat:
# 	for df in [df]:
# 		df[c] = df[c].astype(str).fillna("missing")
# 		df.loc[df[c] == "0", c] = "missing"

print("Missing value imputation done ...")

scaler = QuantileTransformer(output_distribution='normal', random_state=42, n_quantiles=25)
train_data[dense_feature_columns] = scaler.fit_transform(train_data[dense_feature_columns]).astype(np.float32)
eval_data[dense_feature_columns] = scaler.transform(eval_data[dense_feature_columns]).astype(np.float32)

print("Scale done ...")

# train_data['label'] = train_data['sample_weight']
train_data['sample_weight'] = 1 + np.log1p(1 + train_data['sample_weight'])

Missing value imputation done ...
Scale done ...


In [331]:
# train_data['sample_weight'] = 1

In [332]:
train_set, data_info = DatasetFeat.build_trainset(
	train_data=train_data,
	user_col=user_feature_columns,
	item_col=item_feature_columns,
	sparse_col=single_sparse_feature_columns,
	dense_col=dense_feature_columns,
	# multi_sparse_col=multi_sparse_col,
	# pad_val=["missing", "missing"],
	unique_feat=False
)

# train_set, data_info = DatasetFeat.build_trainset(
# 	train_data=train_data,
# 	user_col=user_feature_columns,
# 	item_col=[],
# 	sparse_col=[c for c in single_sparse_feature_columns if c not in item_feature_columns],
# 	dense_col=[c for c in dense_feature_columns if c not in item_feature_columns],
# 	# multi_sparse_col=multi_sparse_col,
# 	# pad_val=["missing", "missing"],
# 	unique_feat=False
# )

eval_set = DatasetFeat.build_testset(eval_data)

print("Transformer dataset generation done ...")

# from recora.data import DatasetPure

# train_set, data_info = DatasetPure.build_trainset(train_data)
# eval_set = DatasetPure.build_evalset(eval_data)

# # # del train_data, eval_data

Transformer dataset generation done ...


In [336]:
import tensorflow as tf

from recora.algorithms import (
	YouTubeRetrieval,
	YouTubeRanking,
	UserCF,
	FM,
	AutoInt,
	WideDeep,
	ItemCF,
	RNN4Rec,
	BPR,
	LightGCN,
	NGCF,
	Transformer,
	DeepFM,
	GraphSage,
	NCF,
)

# ============================================================
# Common settings
# ============================================================

METRICS = [
	"loss",
	"balanced_accuracy",
	"roc_auc",
	"pr_auc",
	"precision",
	"recall",
	"map",
	"ndcg",
]

TF_SESS_CONFIG = {
	"gpu_options": {
		"allow_growth": True,
		"per_process_gpu_memory_fraction": 0.95,
	}
}


def reset_state(model_name: str) -> None:
	tf.compat.v1.reset_default_graph()
	print(f"\n{'=' * 30} {model_name} {'=' * 30}")


def print_hp(hp: dict) -> None:
	print("Hyperparameters:")
	print("-" * 40)
	print(hp)
	print("-" * 40)


# ============================================================
# Model hyperparameters
# ============================================================

MODEL_CONFIGS = {
	"BPR": {
		"MODEL": "BPR",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "random",
		"NUM_NEG": 5,
		"BATCH_SIZE": 4096,
		"LR": 1e-3,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 32,
		"N_EPOCHS": 3,
		"LR_DECAY": True,
	},
	"NCF": {
		"MODEL": "NCF",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 1,
		"BATCH_SIZE": 4096 * (1 + 1),
		"LR": 2e-4,
		"LOSS_TYPE": "focal",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"HIDDEN_UNITS": (512, 256, 128, 64),
		"N_EPOCHS": 5,
		"LR_DECAY": True,
	},
	"DeepFM": {
		"MODEL": "DeepFM",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 1,
		"BATCH_SIZE": 2048 * (1 + 1),
		"LR": 1e-4,
		"LOSS_TYPE": "ranknet",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"HIDDEN_UNITS": (2048, 1024, 512, 256),
		"N_EPOCHS": 5,
		"LR_DECAY": True,
	},
	"Transformer": {
		"MODEL": "Transformer",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
		"NUM_NEG": 1,
		"BATCH_SIZE": 1024 * (1 + 1),
		"LR": 2e-4,
		"LOSS_TYPE": "cross_entropy",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"N_EPOCHS": 3,
		"LR_DECAY": True,
		"NUM_HEAD": 2,
		"NUM_TFM_LAYERS": 1,
        "HIDDEN_UNITS": (256, 128, 64),
		"RECENT_NUM": 5,
		"RANDOM_NUM": None,
	},
	"RNN4Rec": {
		"MODEL": "RNN4Rec",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 1,
		"BATCH_SIZE": 4096,
		"LR": 1e-3,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 8,
		"N_EPOCHS": 3,
		"LR_DECAY": True,
		"RECENT_NUM": 10,
		"HIDDEN_UNITS": (8, 8),
	},
	"YoutubeRanker": {
		"MODEL": "YoutubeRanker",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 1,
		"BATCH_SIZE": 4096 * (1 + 1),
		"LR": 1e-4,
		"LOSS_TYPE": "cross_entropy",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"N_EPOCHS": 5,
		"LR_DECAY": True,
		"RECENT_NUM": 10,
		"RANDOM_NUM": 10,
		"HIDDEN_UNITS": (2048, 1024, 512, 256),
	},
	"LightGCN": {
		"MODEL": "LightGCN",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "popular",
		"NUM_NEG": 1,
		"N_LAYERS": 5,
		"BATCH_SIZE": 2048,
		"LR": 2e-4,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"N_EPOCHS": 5,
		"LR_DECAY": True,
	},
	"NGCF": {
		"MODEL": "NGCF",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "popular",
		"NUM_NEG": 1,
		"N_LAYERS": 3,
		"BATCH_SIZE": 4096,
		"LR": 1e-3,
		"LOSS_TYPE": "bpr",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"LAYER_SIZES": (16, 16, 16),
		"NODE_DROPOUT": 0.1,
		"MESSAGE_DROPOUT": 0.1,
		"N_EPOCHS": 5,
		"LR_DECAY": True,
	},
	"GraphSage": {
		"MODEL": "GraphSage",
		"NEG_SAMPLING": True,
		"NEG_SAMPLER": "unconsumed",
		"NUM_NEG": 1,
		"BATCH_SIZE": 512 * 8,
		"LR": 1e-4,
		"LOSS_TYPE": "softmax",
		"TASK": "ranking",
		"EMBED_SIZE": 16,
		"LAYER_SIZES": (16, 16),
		"HIDDEN_UNITS": (128, 64),
		"neighbor_sampling": True,
		"sample_sizes": (10, 5),
		"N_EPOCHS": 5,
		"LR_DECAY": True,
		"RECENT_NUM": 5,
		"RANDOM_NUM": 5,
	},
}


# ============================================================
# Model builder
# ============================================================

def build_model(model_name: str, hp: dict, data_info):
	reset_state(model_name)

	if model_name == "NCF":
		return NCF(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			reg=1e-6,
			epsilon=1e-8,
			batch_size=hp["BATCH_SIZE"],
			use_bn=False,
			dropout_rate=0.1,
			tf_sess_config=TF_SESS_CONFIG,
			listnet_temperature=0.1,
			num_neg=hp["NUM_NEG"],
		)

	if model_name == 'DeepFM':
		return DeepFM(
            hp["TASK"],
            data_info,
            loss_type=hp["LOSS_TYPE"],
            embed_size=hp["EMBED_SIZE"],
            n_epochs=hp["N_EPOCHS"],
            lr=hp["LR"],
            lr_decay=hp["LR_DECAY"],
            reg=1e-5,
            batch_size=hp["BATCH_SIZE"],
            use_bn=False,
            dropout_rate=0.1,
            hidden_units=hp["HIDDEN_UNITS"],
            tf_sess_config=TF_SESS_CONFIG,
            epsilon=1e-8,
			sampler=hp["NEG_SAMPLER"],
            num_neg=hp["NUM_NEG"],
            approx_ndcg_temperature=0.1,
			listnet_temperature=0.1
        )

	if model_name == "Transformer":
		return Transformer(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			reg=None,
			batch_size=hp["BATCH_SIZE"],
			use_bn=False,
			sampler=hp["NEG_SAMPLER"],
			dropout_rate=None,
			num_heads=hp["NUM_HEAD"],
			num_tfm_layers=hp["NUM_TFM_LAYERS"],
			hidden_units=hp["HIDDEN_UNITS"],
			use_causal_mask=False,
			feat_agg_mode="concat",
			approx_ndcg_temperature=0.1,
			recent_num=hp["RECENT_NUM"],
			random_num=hp["RANDOM_NUM"],
			tf_sess_config=TF_SESS_CONFIG,
			epsilon=1e-7,
		)

	if model_name == "BPR":
		return BPR(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			num_neg=hp["NUM_NEG"],
		)

	if model_name == "RNN4Rec":
		return RNN4Rec(
			task=hp["TASK"],
			data_info=data_info,
			rnn_type="gru",
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			hidden_units=hp["HIDDEN_UNITS"],
			reg=None,
			batch_size=hp["BATCH_SIZE"],
			num_neg=hp["NUM_NEG"],
			dropout_rate=None,
			recent_num=hp["RECENT_NUM"],
			tf_sess_config=TF_SESS_CONFIG,
		)

	if model_name == "YoutubeRanker":
		return YouTubeRanking(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			reg=1e-6,
			batch_size=hp["BATCH_SIZE"],
			num_neg=hp["NUM_NEG"],
			sampler=hp["NEG_SAMPLER"],
			use_bn=False,
			dropout_rate=0.1,
			hidden_units=hp["HIDDEN_UNITS"],
			tf_sess_config=TF_SESS_CONFIG,
			recent_num=hp["RECENT_NUM"],
			random_num=hp["RANDOM_NUM"],
			approx_ndcg_temperature=0.1,
			listnet_temperature=0.1,
            epsilon=1e-8
		)

	if model_name == "LightGCN":
		return LightGCN(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			n_layers=hp["N_LAYERS"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			batch_size=hp["BATCH_SIZE"],
			sampler=hp["NEG_SAMPLER"],
			num_neg=hp["NUM_NEG"],
			tf_sess_config=TF_SESS_CONFIG,
			epsilon=1e-8
		)

	if model_name == "NGCF":
		return NGCF(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			layer_sizes=hp["LAYER_SIZES"],
			node_dropout_rate=hp["NODE_DROPOUT"],
			message_dropout_rate=hp["MESSAGE_DROPOUT"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			batch_size=hp["BATCH_SIZE"],
			num_neg=hp["NUM_NEG"],
			listnet_temperature=1,
			epsilon=1e-8
		)

	if model_name == "GraphSage":
		return GraphSage(
			task=hp["TASK"],
			data_info=data_info,
			loss_type=hp["LOSS_TYPE"],
			embed_size=hp["EMBED_SIZE"],
			layer_sizes=hp["LAYER_SIZES"],
			hidden_units=hp["HIDDEN_UNITS"],
			n_epochs=hp["N_EPOCHS"],
			lr=hp["LR"],
			lr_decay=hp["LR_DECAY"],
			batch_size=hp["BATCH_SIZE"],
			neighbor_sampling=hp["neighbor_sampling"],
			sample_sizes=hp["sample_sizes"],
			num_neg=hp["NUM_NEG"],
			sampler=hp["NEG_SAMPLER"],
			recent_num=hp["RECENT_NUM"],
			random_num=hp["RANDOM_NUM"],
			use_bn=False,
			dropout_rate=None,
			reg=None,
			listnet_temperature=0.05,
			tf_sess_config=TF_SESS_CONFIG,
			epsilon=1e-8
		)

	raise ValueError(f"Unsupported model: {model_name}")

In [ ]:
# ============================================================
# Training entrypoint
# ============================================================

SELECTED_MODEL = "NCF"
hp = MODEL_CONFIGS[SELECTED_MODEL]

print_hp(hp)

model = build_model(SELECTED_MODEL, hp, data_info)

model.fit(
	train_data=train_set,
	neg_sampling=hp["NEG_SAMPLING"],
	verbose=2,
	shuffle=False,
	metrics=METRICS,
	eval_data=eval_set,
	eval_user_num=50000,
	k=TOP_K,
	eval_batch_size=hp["BATCH_SIZE"],
)

print_hp(hp)

Hyperparameters:
----------------------------------------
{'MODEL': 'NCF', 'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 1, 'BATCH_SIZE': 8192, 'LR': 0.0002, 'LOSS_TYPE': 'focal', 'TASK': 'ranking', 'EMBED_SIZE': 16, 'HIDDEN_UNITS': (512, 256, 128, 64), 'N_EPOCHS': 5, 'LR_DECAY': True}
----------------------------------------

============================== NCF ==============================
Training start time: 2026-04-10 01:36:14
With lr_decay, epoch 1 learning rate: 0.00019999999494757503


train: 100%|██████████| 551/551 [00:05<00:00, 97.40it/s] 


Epoch 1 elapsed: 5.657s
	 train_loss: 0.0678


eval_listwise: 100%|██████████| 373/373 [00:07<00:00, 52.98it/s]


	 eval log_loss: 0.6718
	 eval balanced_accuracy: 0.5151
	 eval roc_auc: 0.7245
	 eval pr_auc: 0.7046
	 eval precision@5: 0.1152
	 eval recall@5: 0.1144
	 eval map@5: 0.2295
	 eval ndcg@5: 0.2806
With lr_decay, epoch 2 learning rate: 0.00019199999223928899


train: 100%|██████████| 551/551 [00:05<00:00, 103.15it/s]


Epoch 2 elapsed: 5.357s
	 train_loss: 0.0661


eval_listwise: 100%|██████████| 373/373 [00:06<00:00, 54.43it/s]


In [281]:
data_info.n_items

69

In [282]:
# model = YouTubeRetrieval(
# 	"ranking",
# 	data_info,
# 	loss_type="sampled_softmax",
# 	embed_size=32,
# 	norm_embed=False,
# 	n_epochs=10,
# 	lr=1.25*1e-3,
# 	lr_decay=True,
# 	reg=1e-6,
# 	batch_size=512,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	num_sampled_per_batch=512 * 15,
# 	recent_num=31,
# 	random_num=31
# 	# multi_sparse_combiner='normal'
# )

# Epoch 10 elapsed: 178.519s
# 	 train_loss: 5.8241
# eval_pointwise: 100%|██████████| 3788/3788 [00:01<00:00, 3569.97it/s]
# eval_listwise: 100%|██████████| 5000/5000 [00:05<00:00, 950.57it/s]
# 	 eval log_loss: 0.2397
# 	 eval balanced_accuracy: 0.9053
# 	 eval roc_auc: 0.9718
# 	 eval pr_auc: 0.9684
# 	 eval precision@20: 0.0295
# 	 eval recall@20: 0.2478
# 	 eval map@20: 0.1391
# 	 eval ndcg@20: 0.2134

In [283]:
# model = YouTubeRanking(
# 	"ranking",
# 	data_info,
# 	loss_type="ranknet",
# 	embed_size=32,
# 	n_epochs=5,
# 	lr=2e-3,
# 	lr_decay=False,
# 	reg=1e-6,
# 	batch_size=BATCH_SIZE,
# 	num_neg=NUM_NEG,
# 	sampler=NEG_SAMPLER,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	recent_num=10,
# )
#
# 	 # eval log_loss: 0.3646
# 	 # eval balanced_accuracy: 0.8449
# 	 # eval roc_auc: 0.9204
# 	 # eval pr_auc: 0.7349
# 	 # eval precision@20: 0.0388
# 	 # eval recall@20: 0.1612
# 	 # eval map@20: 0.1497
# 	 # eval ndcg@20: 0.2412


In [284]:
# from libreco.data import DataInfo

# SAVE_PATH = 'youtube-retr'
# SAVE_MODEL_NAME = 'youtuberetrieval'
# # save data_info, specify model save folder
# data_info.save(
# 	path=SAVE_PATH,
# 	model_name=SAVE_MODEL_NAME
# )
# # set manual=True will use `numpy` to save model
# # set manual=False will use `tf.train.Saver` to save model
# # set inference=True will only save the necessary variables for prediction and recommendation
# model.save(
# 	path=SAVE_PATH,
# 	model_name=SAVE_MODEL_NAME,
# 	manual=False,
# 	inference_only=True
# )

In [285]:
data_info.user_col

['polygon_id',
 'recency',
 'length',
 'frequency',
 'monetary',
 'brand_diversity',
 'category_diversity',
 'root_category_diversity',
 'user_avg_ordering_hour',
 'ipo',
 'lpo',
 'aov',
 'avg_item_spending',
 'min_item_spending',
 'max_item_spending',
 'p25_item_spending',
 'p50_item_spending',
 'p75_item_spending',
 'user_discount_ratio',
 'time',
 'order_hour']

In [286]:
import numpy as np

feature_names = scaler.get_feature_names_out()
n_features = len(feature_names)

# create dummy row
x = np.zeros((1, n_features))

# put your value in correct column
idx = list(feature_names).index('order_hour')
x[0, idx] = 8

# transform
x_scaled = scaler.transform(x)

order_hour_scaled = x_scaled[0, idx]
print(order_hour_scaled)

result = model.recommend_user(
	"30953879",
	n_rec=20,
	filter_consumed=False,
	user_feats={
		"order_hour": order_hour_scaled,
	}
)

result = pd.DataFrame(result).rename(columns={"30953879": "category_id"}).astype(int)
brand_category_titles = pd.read_csv('./sample_data/brand_category_titles.csv').drop_duplicates(subset=['category_id'])
result = result.merge(brand_category_titles, on=['category_id'], how='left')
result[['category_id', 'category_title']]

-2.0368341317013883


,category_id,category_title
0,651,سیگار
1,3097,نوشابه
2,92,آب معدنی، طعم دار و گازدار
3,68,خامه
4,72,کشک
5,3070,پنیر صبحانه
6,69,شیر
7,101,نوشیدنی انرژی‌زا
8,96,دوغ
9,3093,ماءالشعیر


In [293]:
result = train_data[train_data['user'] == "30953879"].groupby(['item'])['label'].count().reset_index()
result
# result
# result.merge(brand_category_titles, left_on=['item'] ,right_on=['category_id'], how='left')

,item,label
0,101,17
1,137,1
2,3070,1
3,3073,1
4,3082,1
5,3085,4
6,3091,1
7,3097,1
8,3100,2
9,3101,1


In [89]:
result = df.loc[df['user'] == "30953879"]
result['category_id'] = pd.to_numeric(result['item'], errors='coerce')
brand_category_titles = pd.read_csv('./sample_data/brand_category_titles.csv').drop_duplicates(subset=['category_id'])
result = result.merge(brand_category_titles, on=['category_id'], how='left')
result[['sample_weight', 'category_id', 'category_title']].sort_values(by='sample_weight', ascending=False)

,sample_weight,category_id,category_title
111,0.820335,101,نوشیدنی انرژی‌زا
110,0.820335,62,غذای نیمه آماده منجمد
109,0.742997,101,نوشیدنی انرژی‌زا
108,0.742997,101,نوشیدنی انرژی‌زا
107,0.742997,3082,آدامس و خوشبو کننده دهان
...,...,...,...
4,0.021030,3082,آدامس و خوشبو کننده دهان
3,0.021030,651,سیگار
2,0.017251,3133,کنسرو سبزیجات و حبوبات
1,0.017251,651,سیگار


In [ ]:
import tensorflow as tf
from libreco.data import DataInfo
from libreco.algorithms import TwoTower, YouTubeRetrieval

# important to reset graph if model is loaded in the same shell.
tf.compat.v1.reset_default_graph()
# load data_info
data_info = DataInfo.load(
	SAVE_PATH, model_name=SAVE_MODEL_NAME
)
print(data_info)
# load model, should specify the model name, e.g., DeepFM
model: YouTubeRetrieval = YouTubeRetrieval.load(
	path=SAVE_PATH, model_name=SAVE_MODEL_NAME, data_info=data_info, manual=True
)

In [ ]:
# model.user_embeds_np[data_info.user2id[30953879]].shape
model.user_embeds_np.shape

In [ ]:
import faiss
import numpy as np
import pandas as pd

faiss_index: faiss.swigfaiss.IndexIVFFlat = faiss.read_index('./embed_model/faiss_index.bin')
user_embedding = model.get_user_embedding(30953879, include_bias=True).astype(np.float32).reshape(1, -1)
distances, ids = faiss_index.search(user_embedding, 100)

In [ ]:
distances

In [ ]:
result = pd.DataFrame(list(map(lambda item: list(map(lambda x: int(x), item.split("012"))), [data_info.id2item[x] for x in ids.ravel().tolist()])),
                      columns=['brand_id', 'category_id'])
result['distances'] = distances.ravel()

brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')

In [ ]:
# result[result['category_id'] == 69]
result

# Init `embed_deploy`

In [ ]:
from libserving.serialization import embed2redis, save_embed, save_faiss_index

path = "embed_model"  # specify model saving directory
save_embed(path, model)  # save model in json format

In [ ]:
from libserving.serialization.redis import redis_connection, model_name2redis, id_mapping2redis, user_consumed2redis, user_embed2redis
import ujson
import os

host = "localhost"
port = 6379
db = 0
chunk_size = 10000
with redis_connection(host, port, db) as r:
	model_name2redis(path, r)
	print("\nmodel_name2redis ...")

with redis_connection(host, port, db) as r:
	id_mapping2redis(path, r)
	print("\nid_mapping2redis ...")

with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_consumed.json")) as f:
		data = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, items) in enumerate(data.items()):
		pipe.hset("user_consumed", u, ujson.dumps(items))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_consumed2redis ...")

with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_embed.json")) as f:
		user_embeds = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, embed) in enumerate(user_embeds.items()):
		pipe.hset("user_embed", u, ujson.dumps(embed))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_embed2redis ...")

In [ ]:
from libserving.serialization import save_faiss_index

print(path, model)
save_faiss_index(path, model)

In [ ]:
import requests
import json

payload = {
	"user": "30953879",
	"n_rec": 100
}

result = requests.post(
	"http://127.0.0.1:8000/embed/recommend",
	headers={"Content-Type": "application/json"},
	data=json.dumps(payload)
).json()

In [ ]:
result[['brand_id', 'category_id']] = (
	result['item']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)

# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')

# Init `online_deploy`

In [ ]:
from libserving.serialization import save_online, online2redis

path = "online_model"  # specify model saving directory
save_online(path, model, version=1)  # save model in json format

In [ ]:
from libserving.serialization.redis import redis_connection, model_name2redis, id_mapping2redis
import ujson
import os

host = "localhost"
port = 6379
db = 0
chunk_size = 10000  # ← same as you used before

# ------------------------------------------------------------------
# 1. model_name + id_mapping (small, unchanged)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	model_name2redis(path, r)
	print("\nmodel_name2redis ...")

with redis_connection(host, port, db) as r:
	id_mapping2redis(path, r)
	print("\nid_mapping2redis ...")

# ------------------------------------------------------------------
# 2. user_consumed (your existing chunked version)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_consumed.json")) as f:
		data = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, items) in enumerate(data.items()):
		pipe.hset("user_consumed", u, ujson.dumps(items))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_consumed2redis ...")

# ------------------------------------------------------------------
# 3. features2redis → FULLY CHUNKED (the heavy part for online models)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	feature_path = os.path.join(path, "features.json")
	with open(feature_path) as f:
		feats = ujson.load(f)

	r.set("n_users", feats["n_users"])
	r.set("n_items", feats["n_items"])
	if "max_seq_len" in feats:
		r.set("max_seq_len", feats["max_seq_len"])

	# user_sparse_values (per-user)
	if "user_sparse_col_index" in feats:
		r.hset("feature", "user_sparse", 1)
		r.set("user_sparse_col_index", ujson.dumps(feats["user_sparse_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["user_sparse_values"]):
			pipe.hset("user_sparse_values", str(i), ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# item_sparse_values (list push)
	if "item_sparse_col_index" in feats:
		r.hset("feature", "item_sparse", 1)
		r.set("item_sparse_col_index", ujson.dumps(feats["item_sparse_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["item_sparse_values"]):
			pipe.rpush("item_sparse_values", ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# user_dense_values (per-user)
	if "user_dense_col_index" in feats:
		r.hset("feature", "user_dense", 1)
		r.set("user_dense_col_index", ujson.dumps(feats["user_dense_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["user_dense_values"]):
			pipe.hset("user_dense_values", str(i), ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# item_dense_values (list push)
	if "item_dense_col_index" in feats:
		r.hset("feature", "item_dense", 1)
		r.set("item_dense_col_index", ujson.dumps(feats["item_dense_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["item_dense_values"]):
			pipe.rpush("item_dense_values", ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	print("\nfeatures2redis ...")

# ------------------------------------------------------------------
# 4. user_sparse2redis (small, no chunk needed)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	user_sparse_fields_path = os.path.join(path, "user_sparse_fields.json")
	if os.path.exists(user_sparse_fields_path):
		with open(user_sparse_fields_path) as f:
			user_sparse_fields = ujson.load(f)
		r.hset("user_sparse_fields", mapping=user_sparse_fields)

	user_sparse_idx_mapping_path = os.path.join(path, "user_sparse_idx_mapping.json")
	if os.path.exists(user_sparse_idx_mapping_path):
		with open(user_sparse_idx_mapping_path) as f:
			user_sparse_idx_mapping = ujson.load(f)
		for col, idx_mapping in user_sparse_idx_mapping.items():
			col_name = f"user_sparse_idx_mapping__{col}"
			r.hset(col_name, mapping=idx_mapping)

	print("\nuser_sparse2redis ...")

# ------------------------------------------------------------------
# 5. user_dense2redis (small, no chunk needed)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	user_dense_fields_path = os.path.join(path, "user_dense_fields.json")
	if os.path.exists(user_dense_fields_path):
		with open(user_dense_fields_path) as f:
			user_dense_fields = ujson.load(f)
		r.hset("user_dense_fields", mapping=user_dense_fields)

	print("\nuser_dense2redis ...")

print("\n✅ All online2redis data loaded into Redis with chunking!")

In [ ]:
from libserving.serialization import save_faiss_index

save_faiss_index(path, model)

In [ ]:
result = model.recommend_user(
	30953879,
	n_rec=10,
	filter_consumed=False,
	# user_feats={
	# 	"order_hour": 20
	# }
)

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result[30953879]
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)
# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result

In [ ]:
json.dumps(payload)

In [ ]:
import requests
import json

payload = {
	"user": 30953879,
	"n_rec": 10,
	"filter_consumed": False,
	"return_scores": False,
	"user_feats": {
		"order_hour": 12
	}
}

result = requests.post(
	"http://127.0.0.1:8000/online/recommend",
	headers={"Content-Type": "application/json"},
	data=json.dumps(payload)
).json()

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result['recommendations']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)

# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result

### Read `brand_category_titles` data

In [ ]:
BRAND_CATEGORY_SQL = """
SELECT category_level2_id                                        AS category_id,
       brand_id                                                  AS brand_id,
       dictGet('KF.dim_categories', 'title', category_level2_id) AS category_title,
       dictGet('KF.dim_brands', 'title', brand_id)               AS brand_title
FROM KF.plane_products
GROUP BY category_level2_id, brand_id
"""

brand_category_titles = client.query_df(BRAND_CATEGORY_SQL)

In [ ]:
brand_category_titles.to_csv('brand_category_titles.csv', index=False)

In [ ]:
brand_category_titles = pd.read_csv('brand_category_titles.csv')